## Etapa 1 ) Business Understanding

# Datathon Passos Mágicos — Previsão de Risco de Defasagem Escolar (2024 Holdout)

**Descrição do Case:** Case Passos Mágicos Mudando a vida de crianças e jovens por meio da educação A Associação Passos Mágicos tem uma trajetória de 32 anos de atuação e trabalha na transformação da vida de crianças e jovens de baixa renda, os levando a melhores oportunidades de vida. A transformação, idealizada por Michelle Flues e Dimetri Ivanoff, começou em 1992, atuando dentro de orfanatos no município de Embu-Guaçu. Em 2016, depois de anos de atuação, eles decidem ampliar o programa para que mais jovens tivessem acesso a essa fórmula mágica para transformação que inclui: educação de qualidade, auxílio psicológico/psicopedagógico, ampliação de sua visão de mundo e protagonismo. Passaram então a atuar como um projeto social e educacional, criando assim a Associação Passos Mágicos. A associação busca instrumentalizar o uso da educação como ferramenta para a mudança das condições de vida das crianças e jovens em vulnerabilidade social. Com base no dataset de pesquisa extensiva do desenvolvimento educacional no período de 2022, 2023 e 2024, você tem um desafio de engenharia de Machine Learning para trazer um impacto real na vida dessas crianças. Você pode conhecer mais sobre o projeto aqui. Base de dados e dicionário de dados Você pode encontrar esses materiais aqui. Sobre a entrega Para apoiar a missão de transformar a vida de crianças e jovens por meio da educação, você será responsável por desenvolver um modelo preditivo capaz de estimar o risco de defasagem escolar de cada estudante. Como engenheiro(a) de Machine Learning seu desafio é construir todo o ciclo de vida do modelo, aplicando as melhores práticas de MLOps, desde a construção do melhor modelo até o monitoramento contínuo em produção.


**Objetivo:** construir um modelo preditivo para estimar risco de defasagem escolar.  
**Estratégia temporal:** treinar em **2022 + 2023** (1 registro por aluno/RA) e testar em **2024** (holdout temporal).

> Este notebook está organizado no formato CRISP-DM: Business → Data Understanding → Data Preparation → Modeling → Evaluation.


## Etapa 2 ) Data Understanding 

In [62]:
#2.1 Import de Bibliotecas

In [64]:

import os
import itertools
import numpy as np
import pandas as pd
import joblib

import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_score, recall_score, f1_score,
    precision_recall_curve,
)

import warnings
warnings.filterwarnings(
    "ignore",
    message="Skipping features without any observed values",
)

In [66]:
FILE_PATH = "BASE DE DADOS PEDE 2024 - DATATHON (1).xlsx"  # coloque o arquivo na mesma pasta do script, ou ajuste o caminho aqui
RANDOM_STATE = 42

# 2.2 ) Ler Arquivo e ajustando arquivo

df22 = pd.read_excel(FILE_PATH, sheet_name="PEDE2022")
df23 = pd.read_excel(FILE_PATH, sheet_name="PEDE2023")
df24 = pd.read_excel(FILE_PATH, sheet_name="PEDE2024")

In [67]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import ks_2samp

# =========================
# 2.3.1) Sanity checks (por ano)
# =========================
def quick_quality(df, name, key="RA"):
    out = {}
    out["name"] = name
    out["rows"] = df.shape[0]
    out["cols"] = df.shape[1]
    out["dup_key"] = int(df[key].duplicated().sum()) if key in df.columns else None
    out["missing_%_mean"] = float(df.isna().mean().mean())
    out["missing_%_max"] = float(df.isna().mean().max())
    return out

quality = pd.DataFrame([
    quick_quality(df22, "PEDE2022"),
    quick_quality(df23, "PEDE2023"),
    quick_quality(df24, "PEDE2024")
])

display(quality)

,name,rows,cols,dup_key,missing_%_mean,missing_%_max
0,PEDE2022,860,42,0,0.081838,0.67093
1,PEDE2023,1014,48,0,0.434685,1.00000
2,PEDE2024,1156,50,0,0.379394,1.00000


In [68]:
# interpretação:  ✅ Sem duplicidade de chave → base confiável
#⚠️ Explosão de missing a partir de 2023

#👉 Em 2022:

#base “limpa”
#poucas colunas com muitos nulos

#👉 Em 2023 e 2024:

#quase metade das features tem missing
#várias colunas 100% nulas em determinados anos

#📌 Conclusão-chave:

#O processo de coleta mudou fortemente ao longo do tempo → drift estrutural, não só estatístico.

## Etapa 3 ) Data Preparation

In [70]:
# 3.1 Limpar espaços nos nomes das colunas
for df in (df22,df23,df24):
    df.columns = df.columns.astype(str).str.strip()

In [71]:
# Padronizar nome da coluna de defasagem

df22 = df22.rename(columns={"Defas" : "Defasagem"})
df23 = df23.rename(columns={"Defas" : "Defasagem"})
df24 = df24.rename(columns={"Defas" : "Defasagem"})


In [72]:
# Dropar colunas 100% nulas,verificar outliers, analise de inconsistencia dos dados, analise de volumetria, #analise exploratória em cima da base de treino 

In [73]:
    """
TREINO: 2022 + 2023 (target do próprio ano), com 1 linha por RA (mantém o ano mais recente)
TESTE/HOLDOUT: 2024 (target do próprio ano)
"""

# --- 1) garantir coluna ANO ---
df22["ANO"] = 2022
df23["ANO"] = 2023
df24["ANO"] = 2024

# --- 2) montar treino (2022 + 2023) ---
df_train_full = pd.concat([df22, df23], axis=0, ignore_index=True)

# --- 3) deduplicar por RA no treino (mantém registro mais recente do aluno) ---
df_train = (
    df_train_full
    .sort_values(by=["RA", "ANO"])
    .drop_duplicates(subset="RA", keep="last")
)

# --- 4) dados de teste ---
df_holdout = df24.copy()


In [74]:
# ============================================================
# DATA DRIFT ANALYSIS
# Treino:   df_train  (2022 + 2023)
# Holdout:  df_holdout (2024)
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp

print("\n" + "="*80)
print("DATA DRIFT | df_train  vs  df_holdout")
print("="*80)

# ------------------------------------------------------------
# 1) VOLUMETRIA COMPARATIVA
# ------------------------------------------------------------
print("\n[1] VOLUMETRIA")
print(f"Treino  - linhas: {df_train.shape[0]:,} | colunas: {df_train.shape[1]:,}")
print(f"Holdout - linhas: {df_holdout.shape[0]:,} | colunas: {df_holdout.shape[1]:,}")

# ------------------------------------------------------------
# 2) DRIFT EM FEATURES NUMÉRICAS (KS + MÉDIA)
# ------------------------------------------------------------
print("\n[2] DRIFT NUMÉRICO (KS Test + Δ média)")

num_cols = (
    set(df_train.select_dtypes(include=["number"]).columns)
    & set(df_holdout.select_dtypes(include=["number"]).columns)
)

num_drift = []

for col in sorted(num_cols):
    s_tr = df_train[col].dropna()
    s_te = df_holdout[col].dropna()

    if len(s_tr) < 50 or len(s_te) < 50:
        continue

    ks_stat, ks_p = ks_2samp(s_tr, s_te)

    mean_tr = s_tr.mean()
    mean_te = s_te.mean()
    mean_diff_pct = (mean_te - mean_tr) / (abs(mean_tr) + 1e-9)

    num_drift.append({
        "feature": col,
        "ks_stat": ks_stat,
        "ks_pvalue": ks_p,
        "mean_train": mean_tr,
        "mean_holdout": mean_te,
        "mean_diff_%": mean_diff_pct
    })

num_drift_df = pd.DataFrame(num_drift).sort_values("ks_stat", ascending=False)

print(num_drift_df.head(10).round(4).to_string(index=False))

# ------------------------------------------------------------
# 3) DRIFT EM FEATURES CATEGÓRICAS (PSI simplificado)
# ------------------------------------------------------------
print("\n[3] DRIFT CATEGÓRICO (PSI simplificado)")

cat_cols = (
    set(df_train.select_dtypes(include=["object"]).columns)
    & set(df_holdout.select_dtypes(include=["object"]).columns)
)

def psi_categorical(s_tr, s_te, min_pct=0.01):
    tr_freq = s_tr.value_counts(normalize=True)
    te_freq = s_te.value_counts(normalize=True)

    cats = set(tr_freq.index).union(te_freq.index)

    psi = 0.0
    for c in cats:
        p_tr = tr_freq.get(c, min_pct)
        p_te = te_freq.get(c, min_pct)
        psi += (p_te - p_tr) * np.log(p_te / p_tr)
    return psi

cat_drift = []

for col in sorted(cat_cols):
    s_tr = df_train[col].dropna()
    s_te = df_holdout[col].dropna()

    if len(s_tr) < 50 or len(s_te) < 50:
        continue

    psi = psi_categorical(s_tr, s_te)

    cat_drift.append({
        "feature": col,
        "psi": psi
    })

cat_drift_df = pd.DataFrame(cat_drift).sort_values("psi", ascending=False)

print(cat_drift_df.head(10).round(4).to_string(index=False))

# ------------------------------------------------------------
# 4) CLASSIFICAÇÃO DO NÍVEL DE DRIFT
# ------------------------------------------------------------
print("\n[4] CLASSIFICAÇÃO DO DRIFT")

def classify_ks(ks):
    if ks < 0.10:
        return "baixo"
    elif ks < 0.20:
        return "moderado"
    else:
        return "alto"

def classify_psi(psi):
    if psi < 0.10:
        return "baixo"
    elif psi < 0.25:
        return "moderado"
    else:
        return "alto"

if not num_drift_df.empty:
    num_drift_df["drift_level"] = num_drift_df["ks_stat"].apply(classify_ks)
    print("\nNumérico - distribuição por nível:")
    print(num_drift_df["drift_level"].value_counts().to_string())

if not cat_drift_df.empty:
    cat_drift_df["drift_level"] = cat_drift_df["psi"].apply(classify_psi)
    print("\nCategórico - distribuição por nível:")
    print(cat_drift_df["drift_level"].value_counts().to_string())

# ------------------------------------------------------------
# 5) CONCLUSÃO RESUMIDA (print automático)
# ------------------------------------------------------------
print("\n[5] CONCLUSÃO RESUMIDA")

if not num_drift_df.empty:
    pct_high_num = (num_drift_df["drift_level"] == "alto").mean()
    print(f"Numéricas com drift alto: {pct_high_num:.2%}")

if not cat_drift_df.empty:
    pct_high_cat = (cat_drift_df["drift_level"] == "alto").mean()
    print(f"Categóricas com drift alto: {pct_high_cat:.2%}")

print("\nObservação:")
print("- Drift moderado/alto é esperado em bases temporais.")
print("- O impacto deve ser avaliado no desempenho do holdout (PR-AUC / Recall).")

print("\n" + "="*80)
print("FIM DATA DRIFT")
print("="*80)


DATA DRIFT | df_train  vs  df_holdout

[1] VOLUMETRIA
Treino  - linhas: 1,274 | colunas: 56
Holdout - linhas: 1,156 | colunas: 51

[2] DRIFT NUMÉRICO (KS Test + Δ média)
     feature  ks_stat  ks_pvalue  mean_train  mean_holdout  mean_diff_%
         ANO   1.0000        0.0   2022.7959     2024.0000       0.0006
         IPS   0.3731        0.0      5.5058        6.8297       0.2405
Ano ingresso   0.3019        0.0   2021.2410     2022.5199       0.0006
         IAA   0.2202        0.0      7.1140        8.5436       0.2010
         IPV   0.1877        0.0      7.7591        7.3543      -0.0522
         Ing   0.1861        0.0      6.2003        6.5959       0.0638
         IEG   0.1499        0.0      8.3185        7.3750      -0.1134
         Mat   0.1437        0.0      6.4099        6.2298      -0.0281
     INDE 22   0.1399        0.0      7.0362        7.3683       0.0472
         Por   0.1344        0.0      6.8168        6.1758      -0.0940

[3] DRIFT CATEGÓRICO (PSI simplifica

C:\Users\gtvca\AppData\Local\Temp\ipykernel_2856\1051424156.py:78: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  p_tr = tr_freq.get(c, min_pct)


In [75]:
# ============================================================
# EDA COMPLETA - BASE DE TREINO (df_train)
# - Drop colunas 100% nulas
# - Missing por coluna (top)
# - Cardinalidade (categóricas)
# - Checagens de inconsistência (sanity checks)
# - Outliers (IQR) em numéricas
# - Análise do target (Defasagem) + binário
# ============================================================

In [76]:
# ------------------------------------------------------------
# 1) DROP COLUNAS 100% NULAS
# ------------------------------------------------------------
print("\n[2] COLUNAS 100% NULAS")
null_rate = df_train.isna().mean()
cols_100_null = null_rate[null_rate == 1.0].index.tolist()
print(f"Total colunas 100% nulas: {len(cols_100_null)}")
if cols_100_null:
    print("Amostra/Lista:", cols_100_null[:50], ("..." if len(cols_100_null) > 50 else ""))

# drop seguro (mantém df_train atualizado)
df_train = df_train.drop(columns=cols_100_null)


[2] COLUNAS 100% NULAS
Total colunas 100% nulas: 3
Amostra/Lista: ['Pedra 23', 'INDE 23', 'Destaque IPV.1'] 


In [77]:
# ------------------------------------------------------------
# 2) MISSING POR COLUNA (TOP)
# ------------------------------------------------------------
print("\n[3] TOP MISSING POR COLUNA (%)")
miss = df_train.isna().mean().sort_values(ascending=False)
print((miss.head(20) * 100).round(2).to_string())


[3] TOP MISSING POR COLUNA (%)
Inglês            91.99
Rec Av4           90.82
Portug            79.75
Matem             79.75
Rec Av1           79.59
Ct                79.59
Destaque IDA      79.59
Atingiu PV        79.59
Indicado          79.59
Rec Psicologia    79.59
Destaque IPV      79.59
Rec Av3           79.59
Rec Av2           79.59
Fase ideal        79.59
Destaque IEG      79.59
Cf                79.59
Cg                79.59
Nome              79.59
Ano nasc          79.59
Idade 22          79.59


In [78]:
# ------------------------------------------------------------
# 3) TIPOS E CARDINALIDADE (CATEGÓRICAS)
# ------------------------------------------------------------
print("\n[4] TIPOS (dtypes resumo)")
print(df_train.dtypes.astype(str).value_counts().to_string())

print("\n[4.1] CARDINALIDADE (categóricas) - TOP")
cat_cols = df_train.select_dtypes(include=["object"]).columns.tolist()
if len(cat_cols) == 0:
    print("Nenhuma coluna categórica (object) encontrada.")
else:
    card = df_train[cat_cols].nunique(dropna=True).sort_values(ascending=False)
    print(card.head(20).to_string())

    print("\n[4.2] POSSÍVEL RISCO DE OVERFITTING (cardinalidade muito alta)")
    # heurística: > 30% do número de linhas
    high_card = card[card > 0.30 * len(df_train)]
    if len(high_card) == 0:
        print("Nenhuma coluna categórica com cardinalidade > 30% das linhas.")
    else:
        print(high_card.head(20).to_string())



[4] TIPOS (dtypes resumo)
object     29
float64    21
int64       3

[4.1] CARDINALIDADE (categóricas) - TOP
RA                       1274
Nome Anonimizado         1014
Data de Nasc              909
Nome                      260
Turma                     109
Idade                      34
Fase                       17
Avaliador3                 11
Instituição de ensino      11
Avaliador1                 11
Fase ideal                  9
Fase Ideal                  9
Avaliador2                  8
Rec Av3                     6
Rec Av1                     5
Rec Av4                     5
Gênero                      4
Pedra 20                    4
Pedra 21                    4
Pedra 2023                  4

[4.2] POSSÍVEL RISCO DE OVERFITTING (cardinalidade muito alta)
RA                  1274
Nome Anonimizado    1014
Data de Nasc         909


In [79]:

# ------------------------------------------------------------
# 4) OUTLIERS (IQR) EM COLUNAS NUMÉRICAS
# ------------------------------------------------------------
print("\n[6] OUTLIERS (IQR) - TOP (apenas colunas com >1% outliers)")

num_cols = df_train.select_dtypes(include=["number"]).columns.tolist()
outlier_summary = []

for col in num_cols:
    # ignora colunas quase constantes
    series = df_train[col]
    if series.dropna().nunique() <= 2:
        continue

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    if pd.isna(iqr) or iqr == 0:
        continue

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = int(((series < lower) | (series > upper)).sum())
    pct = outliers / max(len(df_train), 1)

    if pct >= 0.01:
        outlier_summary.append((col, outliers, pct, float(lower), float(upper)))

if len(outlier_summary) == 0:
    print("Nenhuma coluna numérica com >=1% de outliers pelo IQR.")
else:
    outlier_df = pd.DataFrame(
        outlier_summary,
        columns=["coluna", "qtd_outliers", "pct_outliers", "lim_inf_IQR", "lim_sup_IQR"]
    ).sort_values("pct_outliers", ascending=False)

    print(outlier_df.head(15).to_string(index=False))


[6] OUTLIERS (IQR) - TOP (apenas colunas com >1% outliers)
   coluna  qtd_outliers  pct_outliers  lim_inf_IQR  lim_sup_IQR
      IAA           212      0.166405     3.950000    12.350000
      IEG            41      0.032182     4.900000    12.100000
Defasagem            35      0.027473    -2.500000     1.500000
      IPV            32      0.025118     5.227500    10.399500
      IPP            29      0.022763     5.520833     9.687500
  INDE 22            21      0.016484     4.586875     9.649875
      IDA            14      0.010989     1.450000    11.450000


In [80]:
# ------------------------------------------------------------
# 5) TARGET: DEFASAGEM (distribuição, binário e por ano)
# ------------------------------------------------------------
print("\n[7] TARGET: DEFASAGEM")
if "Defasagem" not in df_train.columns:
    print("Coluna 'Defasagem' não encontrada em df_train.")
else:
    s = pd.to_numeric(df_train["Defasagem"], errors="coerce")

    print(f"Missing % (Defasagem): {s.isna().mean():.2%}")
    if s.notna().any():
        print(f"Min: {np.nanmin(s):.4f} | Mediana(p50): {np.nanmedian(s):.4f} | p90: {np.nanpercentile(s.dropna(), 90):.4f} | Max: {np.nanmax(s):.4f}")

    # Label binária (proxy)
    df_train["def_bin"] = (s > 0).astype(int)

    print("\nDistribuição binária (def_bin = Defasagem > 0):")
    print(df_train["def_bin"].value_counts(dropna=False).to_string())

    print("\nProporção binária:")
    print(df_train["def_bin"].value_counts(normalize=True).round(4).to_string())

    if "ANO" in df_train.columns:
        print("\nDefasagem média por ANO:")
        print(df_train.groupby("ANO")["Defasagem"].mean().round(4).to_string())

        print("\nProporção em risco (def_bin) por ANO:")
        print(df_train.groupby("ANO")["def_bin"].mean().round(4).to_string())

print("\n" + "="*80)
print("FIM EDA | df_train")
print("="*80)


[7] TARGET: DEFASAGEM
Missing % (Defasagem): 0.00%
Min: -5.0000 | Mediana(p50): -1.0000 | p90: 0.0000 | Max: 2.0000

Distribuição binária (def_bin = Defasagem > 0):
def_bin
0    1231
1      43

Proporção binária:
def_bin
0    0.9662
1    0.0338

Defasagem média por ANO:
ANO
2022   -1.0615
2023   -0.6548

Proporção em risco (def_bin) por ANO:
ANO
2022    0.0038
2023    0.0414

FIM EDA | df_train


In [81]:
# --- 6) separar X/y do treino ---
y_train = (df_train["Defasagem"] > 0).astype(int)
X_train = df_train.drop(columns=["RA", "Defasagem", "ANO"], errors="ignore")

# --- 7) teste/holdout = 2024 (mantendo nomes X_val/y_val pra não quebrar o resto do notebook) ---
df_holdout = df24.copy()
y_val = (df_holdout["Defasagem"] > 0).astype(int)
X_val = df_holdout.drop(columns=["RA", "Defasagem", "ANO"], errors="ignore")

# --- 8) alinhar colunas (caso 2022/2023/2024 tenham diferenças) ---
all_cols = sorted(set(X_train.columns).union(set(X_val.columns)))
X_train = X_train.reindex(columns=all_cols)
X_val = X_val.reindex(columns=all_cols)

# --- 9) converter colunas "numéricas como texto" (quando possível) ---
for c in X_train.columns:
    X_train[c] = pd.to_numeric(X_train[c], errors="ignore")
    X_val[c] = pd.to_numeric(X_val[c], errors="ignore")

# --- 10) recalcular numéricas e categóricas ---
num_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

# --- 11) correção: coerção numérica nas num_cols (mantendo seu padrão) ---
X_train[num_cols] = X_train[num_cols].apply(pd.to_numeric, errors="coerce")
X_val[num_cols] = X_val[num_cols].apply(pd.to_numeric, errors="coerce")

C:\Users\gtvca\AppData\Local\Temp\ipykernel_2856\1593363374.py:17: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  X_train[c] = pd.to_numeric(X_train[c], errors="ignore")
C:\Users\gtvca\AppData\Local\Temp\ipykernel_2856\1593363374.py:18: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  X_val[c] = pd.to_numeric(X_val[c], errors="ignore")


## Etapa 4 ) Modeling — Pipelines e modelos candidatos

In [98]:
# preprocess para árvore (RF) sem scaler
preprocess_tree = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
        ]), cat_cols),
    ],
    remainder="drop"
)

# preprocess para LogReg - com scaler
preprocess_lr = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_cols),
    ],
    remainder="drop"
)

models = {
    "LogReg": Pipeline(steps=[
        ("prep", preprocess_lr),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            solver="liblinear",
            random_state=RANDOM_STATE
        ))
    ]),
    "RandomForest": Pipeline(steps=[
        ("prep", preprocess_tree),
        ("model", RandomForestClassifier(
            n_estimators=500,
            max_depth=8,
            min_samples_leaf=4,
            max_features="sqrt",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced_subsample"
        ))
    ])
}

In [100]:
print("Treino (linhas):", X_train.shape, " Target:", y_train.shape, " Positivos:", y_train.mean())
print("Holdout 2024 (linhas):", X_val.shape, " Target:", y_val.shape, " Positivos:", y_val.mean())
print("RAs únicos no treino:", df_train["RA"].nunique(), " | linhas no treino:", len(df_train))

Treino (linhas): (1274, 60)  Target: (1274,)  Positivos: 0.033751962323390894
Holdout 2024 (linhas): (1156, 60)  Target: (1156,)  Positivos: 0.1185121107266436
RAs únicos no treino: 1274  | linhas no treino: 1274


In [102]:
import pandas as pd
import numpy as np

def normalize_types_for_sklearn(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()

    # 1) Converter colunas datetime explícitas para string (ISO) - evita mistura no imputador
    dt_cols = X.select_dtypes(include=["datetime64[ns]", "datetime64[ns, UTC]"]).columns
    for c in dt_cols:
        X[c] = X[c].dt.strftime("%Y-%m-%d")

    # 2) Para colunas object "mistas": se houver datetime escondido, converter tudo para string
    obj_cols = X.select_dtypes(include=["object"]).columns
    for c in obj_cols:
        # tenta converter para datetime; se uma parte virar datetime e outra não, vira misto
        parsed = pd.to_datetime(X[c], errors="coerce", infer_datetime_format=True)
        has_some_dt = parsed.notna().any()
        has_some_non_dt = X[c].notna().any() and (~parsed.notna()).any()

        if has_some_dt and has_some_non_dt:
            # mistura de coisas (ex: '2023-01-01' e 'abc' / ou datetime e string)
            X[c] = X[c].astype(str)

    # 3) Garantir que NaN não vire "nan" em strings de forma ruim
    for c in X.columns:
        if X[c].dtype == object:
            X[c] = X[c].where(X[c].notna(), None)

    return X

X_train = normalize_types_for_sklearn(X_train)
X_val   = normalize_types_for_sklearn(X_val)

C:\Users\gtvca\AppData\Local\Temp\ipykernel_2856\227649076.py:16: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  parsed = pd.to_datetime(X[c], errors="coerce", infer_datetime_format=True)
C:\Users\gtvca\AppData\Local\Temp\ipykernel_2856\227649076.py:16: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(X[c], errors="coerce", infer_datetime_format=True)
C:\Users\gtvca\AppData\Local\Temp\ipykernel_2856\227649076.py:16: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-t

## Etapa 5 ) Evaluation

In [105]:
def eval_model(name, pipe, X_tr, y_tr, X_va, y_va, thr=0.5):
    pipe.fit(X_tr, y_tr)
    proba = pipe.predict_proba(X_va)[:, 1]
    pred = (proba >= thr).astype(int)

    pr_auc = average_precision_score(y_va, proba)
    roc_auc = roc_auc_score(y_va, proba)
    prec = precision_score(y_va, pred, zero_division=0)
    rec = recall_score(y_va, pred, zero_division=0)
    f1 = f1_score(y_va, pred, zero_division=0)

    print(f"\n--- {name} ---")
    print(f"PR-AUC: {pr_auc:.4f}  (métrica principal)")
    print(f"ROC-AUC: {roc_auc:.4f}")
    print(f"Prec@0.5:{prec:.4f} | Rec@0.5:{rec:.4f} | F1@0.5:{f1:.4f}")

    return {
        "name": name,
        "pipeline": pipe,
        "pr_auc": pr_auc,
        "roc_auc": roc_auc,
        "precision@0.5": prec,
        "recall@0.5": rec,
        "f1@0.5": f1,
    }

results = []
for name, pipe in models.items():
    results.append(eval_model(name, pipe, X_train, y_train, X_val, y_val, thr=0.5))

best= sorted(results, key =lambda x: x["pr_auc"], reverse=True)[0]
print("\n melhor modelo por PR-AUC:", best["name"], f"(PR-AUC={best['pr_auc']:.4f})")




--- LogReg ---
PR-AUC: 0.3646  (métrica principal)
ROC-AUC: 0.7279
Prec@0.5:0.0000 | Rec@0.5:0.0000 | F1@0.5:0.0000

--- RandomForest ---
PR-AUC: 0.4601  (métrica principal)
ROC-AUC: 0.8884
Prec@0.5:0.6500 | Rec@0.5:0.0949 | F1@0.5:0.1656

 melhor modelo por PR-AUC: RandomForest (PR-AUC=0.4601)


In [107]:
#  """Achar threshold para ter Recall >= 65% e <85% Equilibrio de precision e recall com F1 otimizado (sem função, estilo notebook)."""

In [109]:
import numpy as np
from sklearn.metrics import precision_recall_curve, precision_score, recall_score, f1_score

print("\n==== Threshold ótimo com 65% <= Recall < 85% (fixar threshold) ====")

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_val)[:, 1]

    precision, recall, thresholds = precision_recall_curve(y_val, proba)

    # thresholds tem tamanho len(precision)-1
    precision_t = precision[:-1]
    recall_t = recall[:-1]

    # --- filtro: 0.65 <= recall < 0.85 ---
    valid = np.where((recall_t >= 0.65) & (recall_t < 0.85))[0]

    if len(valid) == 0:
        # fallback: pega o ponto com recall mais próximo do intervalo (0.60..0.80)
        # aqui escolhemos o mais próximo de 0.80 (ajuste se preferir)
        idx = int(np.argmin(np.abs(recall_t - 0.85)))
    else:
        # F1 nos pontos válidos (mais rápido do que chamar f1_score em loop)
        f1_vals = (
            2 * precision_t[valid] * recall_t[valid]
            / (precision_t[valid] + recall_t[valid] + 1e-12)
        )
        idx = valid[np.argmax(f1_vals)]

    thr = float(thresholds[idx])

    # métricas finais nesse threshold
    pred = (proba >= thr).astype(int)
    prec = precision_score(y_val, pred, zero_division=0)
    rec = recall_score(y_val, pred, zero_division=0)
    f1 = f1_score(y_val, pred, zero_division=0)

    print(f"\n=== {name} ===")
    print(
        f"Precision: {prec:.4f} | "
        f"Recall: {rec:.4f} | "
        f"F1: {f1:.4f} | "
        f"Thr: {thr:.6f}"
    )


==== Threshold ótimo com 65% <= Recall < 85% (fixar threshold) ====

=== LogReg ===
Precision: 0.2550 | Recall: 0.6569 | F1: 0.3673 | Thr: 0.001788

=== RandomForest ===
Precision: 0.4167 | Recall: 0.6569 | F1: 0.5099 | Thr: 0.381869


In [205]:
#GridSerach para encontrar melhores hiperparametros para árvore

In [ ]:
from sklearn.model_selection import GridSearchCV, PredefinedSplit

# ------------------------------------------------------------
# Pré-requisitos:
# - models["RandomForest"] -> pipeline do RF com steps: ('prep', ...), ('model', RandomForestClassifier)
# - df_train -> treino (2022+2023) com coluna 'ANO' (2022/2023)
# - X_train, y_train -> features e target do treino
# ------------------------------------------------------------

# 2023 = validação (0), 2022 = treino (-1)
test_fold = [-1 if ano == 2022 else 0 for ano in df_train["ANO"].values]
ps = PredefinedSplit(test_fold=test_fold)

rf_pipe = models["RandomForest"]

# IMPORTANTe: seu step do estimador chama "model" (não "clf")
param_grid = {
    "model__n_estimators": [300, 600, 900],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 3, 5],
    "model__max_features": ["sqrt", "log2", None],
    "model__class_weight": ["balanced", None, "balanced_subsample"],
}

gs = GridSearchCV(
    estimator=rf_pipe,
    param_grid=param_grid,
    scoring="average_precision",  # PR-AUC
    cv=ps,
    n_jobs=-1,
    verbose=2,
)

gs.fit(X_train, y_train)

print("\n====== GRIDSEARCH RANDOM FOREST (best params) ======")
for k, v in gs.best_params_.items():
    print(f"{k}: {v}")

Fitting 1 folds for each of 729 candidates, totalling 729 fits


In [52]:
# ============================================================
# ETAPA FINAL — Export de previsões (2024) + salvar modelo (.pkl)
# - Treina o melhor modelo (best['name']) em TODO o treino (2022+2023)
# - Escolhe threshold no holdout 2024 com restrição: 0.65 <= Recall < 0.85
#   e maximizando F1 (mesma lógica que você usou antes)
# - Exporta df_holdout com proba + classe prevista
# - Salva o pipeline treinado em .pkl (joblib)
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve, precision_score, recall_score, f1_score, average_precision_score
import joblib

# 1) Escolher o melhor modelo (já calculado na Etapa 5)
best_name = best["name"] if "best" in globals() and isinstance(best, dict) and "name" in best else "RandomForest"
pipe_final = models[best_name]

# 2) Treinar em TODO o treino (2022+2023)
pipe_final.fit(X_train, y_train)

# 3) Probabilidades no holdout 2024
proba_2024 = pipe_final.predict_proba(X_val)[:, 1]

# 4) Escolher threshold com 0.65 <= Recall < 0.85 maximizando F1
precision, recall, thresholds = precision_recall_curve(y_val, proba_2024)
precision_t = precision[:-1]
recall_t = recall[:-1]

valid = np.where((recall_t >= 0.65) & (recall_t < 0.85))[0]

if len(valid) == 0:
    # fallback: ponto mais perto do limite superior (0.85)
    idx = int(np.argmin(np.abs(recall_t - 0.85)))
else:
    f1_vals = (2 * precision_t[valid] * recall_t[valid]) / (precision_t[valid] + recall_t[valid] + 1e-12)
    idx = valid[np.argmax(f1_vals)]

final_threshold = float(thresholds[idx])

# 5) Métricas finais no holdout 2024 (com threshold escolhido)
pred_2024 = (proba_2024 >= final_threshold).astype(int)

prec = precision_score(y_val, pred_2024, zero_division=0)
rec  = recall_score(y_val, pred_2024, zero_division=0)
f1   = f1_score(y_val, pred_2024, zero_division=0)
pr_auc = average_precision_score(y_val, proba_2024)

print("\n===== RESULTADO FINAL (HOLDOUT 2024) =====")
print("Modelo final:", best_name)
print(f"PR-AUC: {pr_auc:.4f}")
print(f"Threshold escolhido (fixe em produção): {final_threshold:.6f}")
print(f"Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")

# 6) Export de previsões 2024 (mantém todas as colunas originais + novas colunas)
df_pred_2024 = df_holdout.copy()
df_pred_2024["proba_risco"] = proba_2024
df_pred_2024["pred_risco"] = pred_2024

# (Opcional) coluna com rótulo verdadeiro do holdout
df_pred_2024["y_true"] = y_val.values

# 7) Salvar arquivos
out_csv = "predicoes_2024_com_classificacao.csv"
df_pred_2024.to_csv(out_csv, index=False, encoding="utf-8-sig")

# salvar pipeline treinado
out_pkl = f"modelo_final_{best_name}.pkl"
joblib.dump(pipe_final, out_pkl)

# salvar também threshold em txt (prático para produção)
with open("threshold_final.txt", "w", encoding="utf-8") as f:
    f.write(str(final_threshold))

print("\nArquivos salvos:")
print("-", out_csv)
print("-", out_pkl)
print("-", "threshold_final.txt")



===== RESULTADO FINAL (HOLDOUT 2024) =====
Modelo final: RandomForest
PR-AUC: 0.4601
Threshold escolhido (fixe em produção): 0.381869
Precision: 0.4167 | Recall: 0.6569 | F1: 0.5099

Arquivos salvos:
- predicoes_2024_com_classificacao.csv
- modelo_final_RandomForest.pkl
- threshold_final.txt


In [131]:
FINAL_MODEL = pipe_final   # <- use seu pipeline final
FINAL_THRESHOLD = 0.38186926748858185

In [133]:
# ============================================================
# GERAR FIGURAS PARA O README (salva PNG em reports/figures/)
# ============================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    precision_recall_curve, roc_curve, roc_auc_score, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

FIG_DIR = "reports/figures"
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(name: str):
    path = os.path.join(FIG_DIR, name)
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()
    print("Saved:", path)

# ------------------------------------------------------------
# AJUSTE AQUI (se seus nomes forem diferentes)
# ------------------------------------------------------------
# Se no seu notebook os nomes forem diferentes, troque só aqui:
MODEL = FINAL_MODEL   # <- troque se necessário
THR = final_threshold       # <- troque se necessário

# ------------------------------------------------------------
# PROBAS / PRED no holdout 2024
# ------------------------------------------------------------
proba_2024 = MODEL.predict_proba(X_val)[:, 1]
pred_2024 = (proba_2024 >= THR).astype(int)

# ------------------------------------------------------------
# 1) Distribuição do target (treino e holdout)
# ------------------------------------------------------------
plt.figure()
train_rate = float(np.mean(y_train))
hold_rate  = float(np.mean(y_val))
plt.bar(["train (22+23)", "holdout (24)"], [train_rate, hold_rate])
plt.ylabel("Proporção de positivos (Defasagem > 0)")
plt.title("Distribuição do target (positivos)")
savefig("01_target_rate_train_vs_holdout.png")

# ------------------------------------------------------------
# 2) Curva Precision-Recall + PR-AUC
# ------------------------------------------------------------
prec, rec, thr = precision_recall_curve(y_val, proba_2024)
pr_auc = average_precision_score(y_val, proba_2024)

plt.figure()
plt.plot(rec, prec)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision-Recall (PR-AUC={pr_auc:.4f})")
savefig("02_precision_recall_curve.png")

# ------------------------------------------------------------
# 3) Curva ROC + AUC
# ------------------------------------------------------------
fpr, tpr, _ = roc_curve(y_val, proba_2024)
auc = roc_auc_score(y_val, proba_2024)

plt.figure()
plt.plot(fpr, tpr)
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curve (AUC={auc:.4f})")
savefig("03_roc_curve.png")

# ------------------------------------------------------------
# 4) Matriz de confusão (no threshold final)
# ------------------------------------------------------------
cm = confusion_matrix(y_val, pred_2024)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(values_format="d")
plt.title(f"Matriz de confusão (thr={THR:.4f})")
savefig("04_confusion_matrix.png")

# ------------------------------------------------------------
# 5) Relatório (texto) — salva em arquivo pra colar no README se quiser
# ------------------------------------------------------------
report_txt = classification_report(y_val, pred_2024, digits=4, zero_division=0)
with open(os.path.join(FIG_DIR, "05_classification_report.txt"), "w", encoding="utf-8") as f:
    f.write(report_txt)
print("Saved:", os.path.join(FIG_DIR, "05_classification_report.txt"))

# ------------------------------------------------------------
# 6) Feature importance (se RandomForest e se der pra extrair)
#    *Como você usa ColumnTransformer + OneHot, mapear nomes pode ser chato.
#    Aqui vai uma versão "robusta": tenta pegar os nomes transformados;
#    se falhar, salva apenas top importances sem nomes.
# ------------------------------------------------------------
try:
    # tenta acessar o estimador final (no seu pipeline o step pode ser "model")
    # ajuste o nome do step se necessário:
    if "model" in MODEL.named_steps:
        clf = MODEL.named_steps["model"]
    elif "clf" in MODEL.named_steps:
        clf = MODEL.named_steps["clf"]
    else:
        clf = MODEL.steps[-1][1]

    importances = getattr(clf, "feature_importances_", None)
    if importances is not None:
        # tentar obter nomes das features após o preprocessor
        feat_names = None
        try:
            prep = MODEL.named_steps.get("prep", None)
            if prep is not None and hasattr(prep, "get_feature_names_out"):
                feat_names = prep.get_feature_names_out()
        except Exception:
            feat_names = None

        # monta dataframe
        if feat_names is not None and len(feat_names) == len(importances):
            fi = pd.DataFrame({"feature": feat_names, "importance": importances})
        else:
            fi = pd.DataFrame({"feature": [f"f_{i}" for i in range(len(importances))], "importance": importances})

        fi = fi.sort_values("importance", ascending=False).head(20)

        plt.figure(figsize=(8, 6))
        plt.barh(fi["feature"][::-1], fi["importance"][::-1])
        plt.xlabel("Importance")
        plt.title("Top-20 Feature Importances (RandomForest)")
        savefig("06_feature_importance_top20.png")

        fi.to_csv(os.path.join(FIG_DIR, "06_feature_importance_top20.csv"), index=False, encoding="utf-8")
        print("Saved:", os.path.join(FIG_DIR, "06_feature_importance_top20.csv"))
    else:
        print("Feature importance não disponível neste modelo.")
except Exception as e:
    print("Não foi possível gerar feature importance automaticamente:", repr(e))

print("\n✅ Figuras geradas em:", FIG_DIR)

Saved: reports/figures\01_target_rate_train_vs_holdout.png
Saved: reports/figures\02_precision_recall_curve.png
Saved: reports/figures\03_roc_curve.png
Saved: reports/figures\04_confusion_matrix.png
Saved: reports/figures\05_classification_report.txt
Saved: reports/figures\06_feature_importance_top20.png
Saved: reports/figures\06_feature_importance_top20.csv

✅ Figuras geradas em: reports/figures
